<a href="https://colab.research.google.com/github/bsheese/cs377/blob/18_classification_redesign/18_classification/18_1_Classification_Basics/18_1_0_Classification_Foundations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classification Foundations: Credit Default Prediction

**Author:** Brad Sheese

---

## Why Not Linear Regression for Classification?

Linear regression outputs unbounded numbers. But probabilities must be between 0 and 1.

If we try to use linear regression for classification, the line extends beyond 0-1, suggesting negative probabilities or >100% chances - which makes no sense!

**Solution**: Use the **sigmoid function** to squash outputs between 0 and 1.

### The Sigmoid Function (Logistic)

$$P(Y=1) = \frac{1}{1 + e^{-(\beta_0 + \beta_1X_1 + \dots)}}$$

This 'S-curve' naturally bounds predictions between 0 and 1.

### From Odds to Probability

The sigmoid comes from the **odds ratio**:
- Odds = P / (1-P)
- Log(odds) = β₀ + β₁X₁ + ...
- P = 1 / (1 + e^(-log(odds)))

This means:
- Positive coefficient → higher X → higher probability of class 1
- Negative coefficient → higher X → lower probability of class 1
- Coefficient magnitude → strength of influence

### Learning Objectives
By the end of this notebook, you will:
1. Understand why classification needs special methods
2. Load and explore an imbalanced dataset
3. Train a Logistic Regression classifier
4. Interpret model coefficients
5. Understand probability outputs and the decision threshold

## Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Load the Credit Card Default dataset from OpenML
from sklearn.datasets import fetch_openml

print('Loading dataset...')
data = fetch_openml(name='credit-g', version=1, as_frame=True)
df = data.frame

print(f'Dataset shape: {df.shape}')
print(f'\nFirst few rows:')
df.head()

## Understanding Class Imbalance

**Key insight**: Our dataset has ~70% 'good' credits and ~30% 'bad' defaults.

This imbalance is CRITICAL because:
- A model that predicts 'good' for everyone gets ~70% accuracy!
- But it's completely useless at detecting defaults
- This is called the **Accuracy Paradox**

**How to handle it** (we'll explore in depth):
1. Use class weights to penalize misclassification of minority class
2. Adjust decision threshold (covered in Notebook 3)
3. Use specialized metrics (precision, recall, F1) instead of accuracy

In [ ]:
# Check target variable distribution
print('Target variable distribution:')
class_counts = df['class'].value_counts()
print(class_counts)

bad_pct = (df['class'] == 'bad').mean() * 100
good_pct = (df['class'] == 'good').mean() * 100
print(f'\nPercentage breakdown:')
print(f'  Good: {good_pct:.1f}%')
print(f'  Bad:  {bad_pct:.1f}%')

# Visualize imbalance
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2ecc71', '#e74c3c']
ax.bar(['Good (0)', 'Bad (1)'], [good_pct, bad_pct], color=colors, edgecolor='black')
ax.set_ylabel('Percentage (%)')
ax.set_title('Class Imbalance in Credit Default Dataset')
for i, v in enumerate([good_pct, bad_pct]):
    ax.text(i, v + 1, f'{v:.1f}%', ha='center', fontweight='bold')
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

## Data Preprocessing

We need to:
1. Convert target to binary (1 = default/bad, 0 = good)
2. One-hot encode categorical features
3. Split into train/test sets with stratification
4. Scale features (important for Logistic Regression convergence)

In [ ]:
# Convert target to binary: good = 0, bad = 1
y = (df['class'] == 'bad').astype(int)
X = df.drop(columns=['class'])

# Identify column types
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(include=['number']).columns.tolist()

print(f'Categorical columns: {len(cat_cols)}')
print(f'Numerical columns: {len(num_cols)}')

# One-hot encode
X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)
print(f'\nFeatures after encoding: {X_encoded.shape[1]}')

# Train-test split with stratification to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.3, random_state=42, stratify=y
)

print(f'\nTrain set: {len(X_train)} samples ({y_train.mean()*100:.1f}% bad)')
print(f'Test set: {len(X_test)} samples ({y_test.mean()*100:.1f}% bad)')

## Training Logistic Regression

Logistic Regression uses the **sigmoid** to output probabilities between 0 and 1.

### How It Learns: Maximum Likelihood Estimation

Instead of minimizing squared error (like linear regression), logistic regression:
1. **Maximizes the likelihood** of the observed data
2. Finds coefficients (β₀, β₁, ...) that make the observed labels most probable
3. Uses **gradient descent** to iteratively improve coefficients

### Handling Class Imbalance

We use `class_weight='balanced'` to automatically adjust for imbalance:
- Minority class (bad): gets higher weight
- Majority class (good): gets lower weight
- This penalizes misclassifying the minority class more heavily

In [ ]:
# Scale features (important for convergence)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Logistic Regression WITH class weighting to handle imbalance
model = LogisticRegression(
    max_iter=1000, 
    random_state=42,
    class_weight='balanced'  # Automatically adjust for class imbalance
)
model.fit(X_train_scaled, y_train)

# Get predictions
y_pred = model.predict(X_test_scaled)       # Hard predictions (0 or 1)
y_proba = model.predict_proba(X_test_scaled)[:, 1]  # Soft predictions (probability of default)

print('Model trained with class_weight=balanced!')
print(f'\nFirst 10 predictions:')
print(f"{'Actual':>8} {'Predicted':>10} {'Probability':>12}")
for a, p, prob in zip(y_test.values[:10], y_pred[:10], y_proba[:10]):
    print(f"{a:>8} {p:>10} {prob:>12.3f}")

## Interpreting Model Coefficients

Each coefficient tells us how that feature influences the probability of default:
- **Positive coefficient** → higher feature value → **higher** probability of default
- **Negative coefficient** → higher feature value → **lower** probability of default
- **Magnitude** → strength of the effect (larger = stronger influence)

Note: Because we scaled features, coefficients are comparable - they show the effect of 1 standard deviation change in the feature.

In [ ]:
# Examine coefficients
feature_names = X_encoded.columns.tolist()
coefficients = model.coef_[0]

# Create DataFrame for analysis
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
}).sort_values('Coefficient', key=abs, ascending=False)

print('Top 10 Most Influential Features:')
print(coef_df.head(10).to_string(index=False))

print('\n--- Top 5 that INCREASE default risk ---')
top_increase = coef_df.head(5)
for _, row in top_increase.iterrows():
    print(f"  {row['Feature']}: {row['Coefficient']:.3f}")

print('\n--- Top 5 that DECREASE default risk ---')
top_decrease = coef_df.tail(5).iloc[::-1]
for _, row in top_decrease.iterrows():
    print(f"  {row['Feature']}: {row['Coefficient']:.3f}")

In [ ]:
# Visualize coefficient importance
fig, ax = plt.subplots(figsize=(10, 6))

top_10 = coef_df.head(10)
colors = ['#e74c3c' if c > 0 else '#2ecc71' for c in top_10['Coefficient']]
ax.barh(range(len(top_10)), top_10['Coefficient'], color=colors)
ax.set_yticks(range(len(top_10)))
ax.set_yticklabels(top_10['Feature'])
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
ax.set_xlabel('Coefficient Value')
ax.set_title('Feature Importance: Coefficients (Red=Increase Default Risk, Green=Decrease)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Understanding the Decision Threshold

**Default threshold = 0.5**
- If P(default) >= 0.5, predict 'default' (1)
- If P(default) < 0.5, predict 'good' (0)

This threshold is ARBITRARY - we'll explore tuning it in Notebook 3.

With class_weight='balanced', the model is already adjusted for imbalance, but threshold tuning can further optimize performance.

In [ ]:
# Visualize probability distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Histogram of all probabilities
axes[0].hist(y_proba, bins=40, edgecolor='black', alpha=0.7)
axes[0].axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Threshold=0.5')
axes[0].set_xlabel('Predicted Probability of Default')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Predicted Probabilities')
axes[0].legend()

# Right: Overlapping histograms by actual class
axes[1].hist(y_proba[y_test==0], bins=30, alpha=0.7, label='Actual: Good (0)', color='green', edgecolor='black')
axes[1].hist(y_proba[y_test==1], bins=30, alpha=0.7, label='Actual: Bad (1)', color='red', edgecolor='black')
axes[1].axvline(x=0.5, color='black', linestyle='--', linewidth=2, label='Threshold=0.5')
axes[1].set_xlabel('Predicted Probability of Default')
axes[1].set_ylabel('Count')
axes[1].set_title('Probability Distribution by Actual Class')
axes[1].legend()

# Add annotation about separation
axes[1].annotate('Model assigns higher\nprobabilities to actual defaults', 
                 xy=(0.7, 10), xytext=(0.55, 40),
                 fontsize=10, ha='center',
                 arrowprops=dict(arrowstyle='->', color='gray'))

plt.tight_layout()
plt.show()

## Key Takeaways

1. **Classification needs sigmoid** to bound probabilities between 0-1
2. **Odds ratio interpretation**: Coefficients represent log(odds) - positive = increases default probability
3. **class_weight='balanced'** automatically handles class imbalance
4. **Feature importance** from coefficients helps interpret the model
5. **Default threshold = 0.5** but can be tuned based on business costs
6. **Accuracy alone is misleading** on imbalanced data

In Notebook 2, we'll learn how to properly evaluate classifiers using the confusion matrix and other metrics.

In Notebook 3, we'll explore threshold tuning in depth.